In [ ]:
# ---
# jupyter:
#   jupytext:
#     text_representation:
#       extension: .py
#       format_name: percent
#       format_version: '1.3'
#       jupytext_version: 1.19.6
#   kernelspec:
#     display_name: Python (latest Jaspy)
#     language: python
#     name: jaspy
# ---

# AAA_summary: synthetic study across all 10 designs

Reads the saved artifacts of `AAA01` to `AAA10` (nothing is refitted) and draws the error measures of the networks for every design (CVPE, Kendall's tau(w), lower tail, upper tail and the Copula Energy Score). Each of the five measures gives **one full-page figure** with 5 rows (Student-t, Student-t with a nonmonotonic tau(w), Gumbel, Clayton, Gumbel-Clayton mixture) and 2 columns (weak and strong tau(w)). **All panels of one figure share one y-axis**, so every bar can be compared directly. Bars are means over the data draws, error bars 95% t-intervals across draws; conditioning variants are light grey (w only), mid grey (spline basis), dark (FiLM), the unconditional CODINE is unfilled. Every design uses `synthetic_study.N_TRAIN` training points per draw (2,000 by default; the environment variable `THESIS_SYNTH_N` overrides it).

Figures are written to `Images AAA_summary`. Designs whose notebook has not been run are drawn as empty panels marked "not run".

Three reference bars are drawn next to the networks in every figure: the Acar kernel estimator of the family that generated the data (the best-fitting of the seven Acar models in the two mixture designs; a best case, since the family is known), the kernel estimator of Gijbels et al. (2011) and the empirical checkerboard Bernstein copula (ECBC) of Lu and Ghosh (2024). Each can be turned off for all figures with `SHOW_ACAR_MATCHED`, `SHOW_GIJBELS` and `SHOW_ECBC` in `synthetic_report.py`.

The two sections after the figures check the marginal uniformity of the stored networks and relate it to their accuracy. The last section collects the benchmark numbers quoted in the thesis text and writes the table of the synthetic overview and the table of the networks against the kernel benchmark. Every table is written as LaTeX and CSV.

## Load the 10 artifacts

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt

import plotting_core
import synthetic_report as R
from run_support import load_artifact

plotting_core.configure("AAA_summary")
plt.rcParams.update({"font.size": 12})

LONG, GIJ, ECBC, missing = {}, {}, {}, []
for d in R.all_designs():
    art = load_artifact(d['artifact'])
    if art and art.get('summary_by_draw') is not None:
        LONG[d['tag']] = art['summary_by_draw']
        GIJ[d['tag']] = art.get('gijbels')      # saved by AAA01-AAA10; nothing is refitted
        ECBC[d['tag']] = art.get('ecbc')
        if art.get('smoke'):
            print("WARNING: smoke artifact", d['artifact'])
    else:
        LONG[d['tag']] = None; missing.append(d['notebook'])
print(f"{sum(v is not None for v in LONG.values())} of 10 designs available" + (f"; missing: {missing}" if missing else ""))
print("data draws per design:", {t: int(v['rep'].nunique()) for t, v in LONG.items() if v is not None})

## 1. CVPE (excess over the oracle) of the networks

Suggested caption: *CVPE (excess over the oracle) of the networks, by design (rows: DGP family; columns: weak and strong tau(w)); one common y-axis for all panels; mean and 95% interval across data draws. The blue bar is the Acar kernel estimator of the family that generated the data (the best-fitting of the seven Acar models in the mixture rows; a best case), the tan bar the kernel estimator of Gijbels et al. (2011) and the green bar the empirical checkerboard Bernstein copula of Lu and Ghosh (2024); the excess of the Gijbels and ECBC bars is computed on a 32 x 32 cell discretisation and is a reference level, not a like-for-like CVPE.*

In [ ]:
fig, ylim = R.summary_page_figures(LONG, 'cvpe', 'summary_cvpe', gij_by_tag=GIJ, ecbc_by_tag=ECBC)
print('common y-axis:', tuple(round(v, 5) for v in ylim))
plt.show()

## 2. Kendall's tau(w) RMSE

Suggested caption: *Kendall's tau(w) RMSE, by design (rows: DGP family; columns: weak and strong tau(w)); one common y-axis for all panels; mean and 95% interval across data draws. The blue bar is the Acar kernel estimator of the family that generated the data (the best-fitting of the seven Acar models in the mixture rows; a best case), the tan bar the kernel estimator of Gijbels et al. (2011) and the green bar the empirical checkerboard Bernstein copula of Lu and Ghosh (2024).*

In [ ]:
fig, ylim = R.summary_page_figures(LONG, 'tau', 'summary_tau', gij_by_tag=GIJ, ecbc_by_tag=ECBC)
print('common y-axis:', tuple(round(v, 5) for v in ylim))
plt.show()

## 3. Lower tail dependence RMSE, chi(q = 0.95)

Suggested caption: *Lower tail dependence RMSE, chi(q = 0.95), by design (rows: DGP family; columns: weak and strong tau(w)); one common y-axis for all panels; mean and 95% interval across data draws. The blue bar is the Acar kernel estimator of the family that generated the data (the best-fitting of the seven Acar models in the mixture rows; a best case), the tan bar the kernel estimator of Gijbels et al. (2011) and the green bar the empirical checkerboard Bernstein copula of Lu and Ghosh (2024).*

In [ ]:
fig, ylim = R.summary_page_figures(LONG, 'lower_tail', 'summary_lower_tail', gij_by_tag=GIJ, ecbc_by_tag=ECBC)
print('common y-axis:', tuple(round(v, 5) for v in ylim))
plt.show()

## 4. Upper tail dependence RMSE, chi(q = 0.95)

Suggested caption: *Upper tail dependence RMSE, chi(q = 0.95), by design (rows: DGP family; columns: weak and strong tau(w)); one common y-axis for all panels; mean and 95% interval across data draws. The blue bar is the Acar kernel estimator of the family that generated the data (the best-fitting of the seven Acar models in the mixture rows; a best case), the tan bar the kernel estimator of Gijbels et al. (2011) and the green bar the empirical checkerboard Bernstein copula of Lu and Ghosh (2024).*

In [ ]:
fig, ylim = R.summary_page_figures(LONG, 'upper_tail', 'summary_upper_tail', gij_by_tag=GIJ, ecbc_by_tag=ECBC)
print('common y-axis:', tuple(round(v, 5) for v in ylim))
plt.show()

## 5. Copula Energy Score (excess over the oracle)

Suggested caption: *Excess Copula Energy Score over the oracle, by design (rows: DGP family; columns: weak and strong tau(w)); one common y-axis for all panels; mean and 95% interval across data draws; raw (not margin-calibrated) score, lower is better. The blue bar is the Acar kernel estimator of the family that generated the data (the best-fitting of the seven Acar models in the mixture rows; a best case), the tan bar the kernel estimator of Gijbels et al. (2011) and the green bar the empirical checkerboard Bernstein copula of Lu and Ghosh (2024).*

If this cell raises a `KeyError` saying that the column `excess_ces` is missing, the Copula Energy Score was not stored in the saved frames of the design notebooks and they have to be rerun.

In [ ]:
fig, ylim = R.summary_page_figures(LONG, 'ces', 'summary_ces', gij_by_tag=GIJ, ecbc_by_tag=ECBC)
print('common y-axis:', tuple(round(v, 5) for v in ylim))
plt.show()

## Marginal-uniformity check of the stored synthetic networks

Same check and the same gate as in the real-data pipeline (`ks_n_samples` draws per covariate value, 1,000 by default, KS statistic against Uniform(0,1), D_max against the `gate_config` threshold), applied to the networks stored by AAA01 to AAA10. Nothing is retrained.

In [ ]:
# ── Marginal-uniformity check of the STORED synthetic networks (no retraining) ─────────────────────────────────────
# Same check and same gate as the real-data pipeline (fold_pipeline.neural_job): `ks_n_samples` draws per covariate value,
# KS statistic of both margins against Uniform(0,1), D_max compared with the gate_config threshold.
#   C-CODINE / C-CopFlow : 7 covariate percentiles (GC.KS_PERCENTILES) of the draw's training w; grid sampler for C-CODINE.
#   CODINE (unconditional, split halves): one check (grid sampler), as in the real-data pipeline.
# Reads <FP.BATCH_DIR>/synthetic/states/*.pt, the same files the CES stage of AAA01-AAA10 reloaded.
import numpy as np
import pandas as pd
import torch
from IPython.display import display

import fold_pipeline as FP
import fold_models as MS
import synthetic_study as S
import synthetic_report as R
import model_evaluation as ME
import gate_config as GC
import reporting_tables as RT
from codine import make_codine_sample_fn, sample_from_codine
from conditional_copula_flow import make_flow_sample_fn
from run_support import load_artifact, SMOKE

torch.set_num_threads(1)
RT.configure("AAA_summary")
BATCH = FP.BATCH_DIR / "synthetic"
KS_N = FP.default_hyper()["ks_n_samples"]               # draws per covariate value, as in the real-data gate
SEED = 0                                                 # TRAIN_SEEDS = (0,): the one training seed of every draw

MODELS = ([(FP.UNCOND_CODINE, "codine_uncond", "CODINE")]
          + [(f"{FP.SPLIT_CODINE}|{r}", "codine_regime", "CODINE") for r in ("low", "high")]
          + [(m, "ccodine", "C-CODINE") for m in FP.CCODINE_VARIANTS]
          + [(m, "ccopflow", "C-CopFlow") for m in FP.CCOPFLOW_VARIANTS])

rows, skipped = [], []
for d in R.all_designs():
    art = load_artifact(d["artifact"])
    if not art or (art.get("smoke") and not SMOKE):      # a smoke artifact is only used by a smoke run
        skipped.append(d["tag"]); continue
    spec = S.DGPS[d["dgp"]]
    for rep in range(int(art["mc_reps"])):
        # training covariate of this draw, identical to bundle['w'] of make_bundle
        w_tr = S.covariate_stream(int(art["n_train"]), S.DATA_SEED + rep)[1]
        w_chk = np.quantile(w_tr, GC.KS_PERCENTILES)
        for model, kind, family in MODELS:
            key = S.synth_key(dict(bundle=dict(dgp=d["dgp"], rep=rep), seed=SEED, model=model))
            try:
                net, _ = MS.load_fitted(BATCH, key)
            except MS.NotReconstructable as exc:
                rows.append(dict(design=d["tag"], rep=rep, model=model, family=family, d_max=np.nan, n_tests=0,
                                 passed=False, note=str(exc)[:80])); continue
            if kind in ("codine_uncond", "codine_regime"):
                fn = lambda _wv, n, sd, o=net: sample_from_codine(o, n_samples=n, seed=sd)
                wv, grid = [0.5], True
            elif kind == "ccodine":
                fn, wv, grid = make_codine_sample_fn(net), w_chk, True
            else:
                fn, wv, grid = make_flow_sample_fn(net), w_chk, False
            with torch.no_grad(), R.quiet():
                gate = ME.ks_gate_summary(ME.marginal_uniformity_check(fn, wv, n_samples=KS_N, seed=100 * SEED), grid=grid)
            rows.append(dict(design=d["tag"], rep=rep, model=model, family=family, d_max=gate["d_max"],
                             n_tests=gate["n_tests"], passed=gate["pass"], note=""))
    print("done", d["tag"])

KS_SYN = pd.DataFrame(rows, columns=["design", "rep", "model", "family", "d_max", "n_tests", "passed", "note"])
print("skipped (missing or smoke artifact):", skipped, "| networks checked:", int((KS_SYN.n_tests > 0).sum()),
      "| not reconstructable:", int((KS_SYN.n_tests == 0).sum()))

KS_SUM = (KS_SYN[KS_SYN.n_tests > 0].groupby(["design", "family"], sort=False)
          .agg(nets=("passed", "size"), passing=("passed", "sum"), median_Dmax=("d_max", "median"), max_Dmax=("d_max", "max")))
display(KS_SUM.round(3))
ALL = (KS_SYN[KS_SYN.n_tests > 0].groupby("family")
       .agg(nets=("passed", "size"), passing=("passed", "sum"), share=("passed", "mean"), median_Dmax=("d_max", "median"), max_Dmax=("d_max", "max")))
display(ALL.round(3))
if (KS_SYN.n_tests > 0).any():
    R.synthetic_uniformity_table(KS_SYN)
    R.synthetic_uniformity_design_table(KS_SYN)

## Does a larger uniformity statistic go with worse accuracy?

Within the same design and network type, rank correlation between each network's D_max and its error measures.

In [ ]:
# ── Does a larger marginal-uniformity statistic go with worse accuracy? (uses KS_SYN and LONG from the cells above; no refitting)
# Each conditional network is one row: its D_max (this notebook) joined with its own excess CVPE, tau(w) RMSE, tail RMSEs and raw excess CES
# (summary_by_draw of the artifact). Design and architecture are removed by subtracting the mean of the (design, model) cell, so the
# question is: within the same design and the same network type, do draws with a larger D_max have larger errors?
from scipy.stats import spearmanr

METRICS = [("excess", "excess CVPE"), ("tau_rmse", "tau(w) RMSE"), ("tail_ll", "lower-tail RMSE"),
           ("tail_uu", "upper-tail RMSE"), ("excess_ces", "excess CES (raw)")]
cond = KS_SYN[KS_SYN.family.isin(["C-CODINE", "C-CopFlow"]) & (KS_SYN.n_tests > 0)]
parts = []
for tag, L in LONG.items():
    if L is None:
        continue
    keep = ["model", "rep"] + [c for c, _ in METRICS if c in L.columns]
    parts.append(cond[cond.design == tag].merge(L[keep], on=["model", "rep"], how="inner"))
J = pd.concat(parts, ignore_index=True) if parts else cond.iloc[:0]
print(len(J), "of", len(cond), "conditional network runs joined")

cell = ["design", "model"]
rows = []
for fam, sub in J.groupby("family"):
    dd = sub.d_max - sub.groupby(cell).d_max.transform("mean")
    for c, lab in METRICS:
        if c not in sub or sub[c].isna().all():
            continue
        mm = sub[c] - sub.groupby(cell)[c].transform("mean")
        rho, p = spearmanr(dd, mm, nan_policy="omit")
        fail = ~sub.passed.values
        rows.append(dict(family=fam, metric=lab, runs=len(sub), failing=int(fail.sum()),
                         spearman_within=rho, p_value=p,
                         median_dev_failing=float(np.nanmedian(mm[fail])) if fail.any() else np.nan,
                         median_dev_passing=float(np.nanmedian(mm[~fail])) if (~fail).any() else np.nan))
KS_ACC = pd.DataFrame(rows)
display(KS_ACC.round(4))
if len(KS_ACC):
    R.synthetic_uniformity_accuracy_table(KS_ACC, {f: s.groupby(cell).ngroups for f, s in J.groupby("family")})
print("Reading: spearman_within > 0 means a larger D_max goes with a larger error inside the same design and network type. "
      "median_dev_* is the median deviation of the error from that cell's mean, for failing and passing runs.")

## 6. Benchmark numbers and the tables of the synthetic study

One row per design: mean and standard deviation across the data draws of the Gijbels et al. (2011) benchmark (excess CVPE, excess and margin-calibrated Copula Energy Score, tau(w) RMSE, tail RMSEs, bandwidths, minimum effective sample size) and, next to it, the mean over draws of the same error measures for the nine networks and the unconditional CODINE. Read from the saved artifacts, nothing is refitted. Saved as `synthetic_benchmark_numbers` (LaTeX and CSV) in `Tables AAA_summary`.

In [ ]:
BENCH = R.benchmark_numbers_table(load_artifact)
RT.write_table(BENCH, "synthetic_benchmark_numbers", index=False,
               caption="Benchmark numbers per design: mean and SD over the data draws of the kernel benchmark and mean of the networks.")
display(BENCH.iloc[:, :12].round(5))
R.synthetic_overview_table(load_artifact)
R.networks_vs_gijbels_table(load_artifact)